# Step 2 â€” CDS extraction, splicing validation, per-exon hashing (chr22, reference proteome)

Anchors on **CDS** (not gene genomic span), per the handoff. Validates two cross-level invariants against
GENCODE v46 chr22 before trusting the per-exon hashes:

- `sum(GTF CDS-exon lengths) == len(CDS from pc_transcripts.fa) - 3` (see note below on the off-by-3)
- `translate(CDS) == protein` (from `pc_translations.fa`)

Uses only the already-downloaded, MD5-verified GENCODE files (`data/reference/`) -- no raw genome FASTA
needed, because `pc_transcripts.fa` headers already carry transcript-relative `CDS:start-end` coordinates.

In [1]:
import gzip
import re
import hashlib
import base64
from collections import defaultdict, Counter

REF = "../data/reference"
GTF = f"{REF}/gencode.v46.basic.annotation.gtf.gz"
PC_TRANSCRIPTS = f"{REF}/gencode.v46.pc_transcripts.fa.gz"
PC_TRANSLATIONS = f"{REF}/gencode.v46.pc_translations.fa.gz"
CHROM = "chr22"

In [2]:
# hash functions from notebook 01 (protein normalize/digest) plus a CDS-nucleotide variant.
# CDS DECISION (documented per handoff \"MANDATORY... decide stop-codon inclusion\"):
# we hash the CDS WITHOUT the trailing stop codon, matching the protein-layer decision
# (stop encodes no residue) so that len(hashed_cds_nt) == 3 * len(hashed_protein) always.

def md5_digest(seq: str) -> str:
    return hashlib.md5(seq.encode("ascii")).hexdigest()

def ga4gh_sq_digest(seq: str) -> str:
    digest = hashlib.sha512(seq.encode("ascii")).digest()[:24]
    return "SQ." + base64.urlsafe_b64encode(digest).decode("ascii").rstrip("=")

CODON_TABLE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def translate(nt: str) -> str:
    # standard genetic code only; selenocysteine (in-frame recoded UGA) will show up
    # as a premature '*' here -- that's expected and handled as a flagged category below,
    # not patched over, since correct Sec recoding needs the SECIS element, not just the codon.
    codons = (nt[i:i+3] for i in range(0, len(nt) - len(nt) % 3, 3))
    return "".join(CODON_TABLE.get(c, "X") for c in codons)

## Parse chr22 CDS features from the GTF

GTF `CDS` features **exclude** the stop codon (it's a separate `stop_codon` feature) -- confirmed on this
file (see invariant below). CDS blocks are ordered into transcript (5'->3') order using strand: ascending
genomic start for `+`, descending for `-`, since `pc_transcripts.fa` sequences are already given in mRNA sense.

In [3]:
attr_re = re.compile(r'(\w+) "([^"]*)"')

def parse_attrs(field):
    return dict(attr_re.findall(field))

def parse_tags(field):
    return set(v for k, v in attr_re.findall(field) if k == "tag")

transcripts = defaultdict(lambda: {"strand": None, "cds": [], "gene_id": None, "tags": set(), "level": None})

with gzip.open(GTF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if f[0] != CHROM or f[2] not in ("CDS", "transcript"):
            continue
        attrs = parse_attrs(f[8])
        tid = attrs["transcript_id"]
        t = transcripts[tid]
        t["strand"] = f[6]
        t["gene_id"] = attrs["gene_id"]
        if f[2] == "CDS":
            t["cds"].append((int(f[3]), int(f[4])))
        else:
            t["level"] = attrs.get("level")
            t["tags"] = parse_tags(f[8])

for t in transcripts.values():
    t["cds"].sort(key=lambda se: se[0], reverse=(t["strand"] == "-"))

chr22_cds_transcripts = {tid: t for tid, t in transcripts.items() if t["cds"]}
print(f"chr22 transcripts with CDS entries: {len(chr22_cds_transcripts)}")

chr22 transcripts with CDS entries: 1404


## Load the reference proteome FASTAs

Both files are whole-genome (all chromosomes); we index by transcript ID and only look up chr22 IDs later,
so no separate chr22-only file needs to be written for this step.

In [4]:
def load_transcripts_fasta(path):
    """pc_transcripts.fa: header field 0 = ENST id; header carries CDS:start-end (1-based, transcript-relative)."""
    seqs, meta = {}, {}
    tid, chunks = None, []
    with gzip.open(path, "rt") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if tid is not None:
                    seqs[tid] = "".join(chunks)
                fields = line[1:].split("|")
                tid = fields[0]
                meta[tid] = fields
                chunks = []
            else:
                chunks.append(line)
        if tid is not None:
            seqs[tid] = "".join(chunks)
    return seqs, meta

def load_translations_fasta(path):
    """pc_translations.fa: header field 1 = ENST id (field 0 is the ENSP protein id)."""
    seqs = {}
    tid, chunks = None, []
    with gzip.open(path, "rt") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if tid is not None:
                    seqs[tid] = "".join(chunks)
                tid = line[1:].split("|")[1]
                chunks = []
            else:
                chunks.append(line)
        if tid is not None:
            seqs[tid] = "".join(chunks)
    return seqs

tx_seqs, tx_meta = load_transcripts_fasta(PC_TRANSCRIPTS)
prot_seqs = load_translations_fasta(PC_TRANSLATIONS)
print(f"pc_transcripts.fa: {len(tx_seqs)} transcripts (genome-wide)")
print(f"pc_translations.fa: {len(prot_seqs)} proteins (genome-wide)")

pc_transcripts.fa: 111868 transcripts (genome-wide)
pc_translations.fa: 111868 proteins (genome-wide)


## Extract, validate, and hash

Two things are handled as **expected categories, not bugs**, per the handoff's "treat as QC flag" philosophy:

1. **The off-by-3.** GTF `CDS` features exclude the stop codon; `pc_transcripts.fa`'s `CDS:` span includes it.
   This is the handoff's documented "#1 silent error" and shows up as a consistent 3nt offset -- not a real
   mismatch once accounted for.
2. **Non-AUG initiator override.** A few transcripts (tagged `non_ATG_start`) use a near-cognate start codon
   (e.g. `CTG`) that the initiator tRNA still reads as Met in vivo. GENCODE/UniProt encode the canonical
   protein with `M` at position 0 regardless -- a naive codon-table translation will show the literal
   (non-Met) residue instead. We accept the match if only position 0 differs and the tag confirms it.

In [5]:
def extract_cds_span(fields):
    for f in fields:
        if f.startswith("CDS:"):
            s, e = f[4:].split("-")
            return int(s), int(e)
    return None

catalog = []      # successfully validated entries, ready for the SQLite catalog (next notebook)
flagged = []       # (transcript_id, reason) -- kept as metadata, not discarded

present_ids = [tid for tid in chr22_cds_transcripts if tid in tx_seqs and tid in prot_seqs]

for tid in present_ids:
    t = chr22_cds_transcripts[tid]
    span = extract_cds_span(tx_meta[tid])
    if span is None:
        flagged.append((tid, "no_cds_span"))
        continue
    start, end = span
    cds_with_stop_maybe = tx_seqs[tid][start-1:end].upper()

    exon_lens = [e - s + 1 for s, e in t["cds"]]
    if sum(exon_lens) != len(cds_with_stop_maybe) - 3:
        flagged.append((tid, "length_mismatch"))
        continue

    protein = prot_seqs[tid].upper()
    if len(cds_with_stop_maybe) == 3 * (len(protein) + 1):
        coding_part = cds_with_stop_maybe[:-3]
    elif len(cds_with_stop_maybe) == 3 * len(protein):
        coding_part = cds_with_stop_maybe
    else:
        flagged.append((tid, "unexpected_length_ratio"))
        continue

    translated = translate(coding_part)
    non_atg = "non_ATG_start" in t["tags"]
    if translated != protein:
        if non_atg and protein[0] == "M" and translated[1:] == protein[1:]:
            pass  # accepted: biological initiator override, see markdown above
        else:
            flagged.append((tid, "translate_mismatch"))
            continue

    exon_chunks, pos = [], 0
    for s, e in t["cds"]:
        n = e - s + 1
        exon_chunks.append(coding_part[pos:pos+n])
        pos += n

    catalog.append({
        "transcript_id": tid,
        "gene_id": t["gene_id"],
        "protein_seq": protein,
        "protein_md5": md5_digest(protein),
        "protein_sq": ga4gh_sq_digest(protein),
        "cds_seq": coding_part,
        "cds_md5": md5_digest(coding_part),
        "cds_sq": ga4gh_sq_digest(coding_part),
        "exon_hashes": [(md5_digest(c), ga4gh_sq_digest(c), len(c)) for c in exon_chunks],
        "level": t["level"],
        "tags": sorted(t["tags"]),
    })

print(f"chr22 CDS-transcripts considered: {len(present_ids)}")
print(f"validated (in catalog):          {len(catalog)}")
print(f"flagged (excluded, categorized): {len(flagged)}")

chr22 CDS-transcripts considered: 1398
validated (in catalog):          1341
flagged (excluded, categorized): 57


In [6]:
# categorize the flagged set using GTF tags -- confirms every exclusion has a known biological cause
# rather than being an unexplained parsing bug
cat_counter = Counter()
for tid, reason in flagged:
    tg = chr22_cds_transcripts[tid]["tags"]
    if "seleno" in tg:
        cat_counter["selenoprotein (internal recoded stop, e.g. TXNRD2)"] += 1
    elif tg & {"cds_start_NF", "cds_end_NF", "mRNA_start_NF", "mRNA_end_NF"}:
        cat_counter["incomplete CDS (NF tags, e.g. IG/TR gene segments)"] += 1
    else:
        cat_counter[f"unexplained ({reason})"] += 1

for cat, n in cat_counter.most_common():
    print(f"  {n:3d}  {cat}")

   47  incomplete CDS (NF tags, e.g. IG/TR gene segments)
   10  selenoprotein (internal recoded stop, e.g. TXNRD2)


## Worked example

One MANE_Select transcript, showing protein + whole-CDS + per-exon hashes end to end.

In [7]:
example = next(e for e in catalog if "MANE_Select" in e["tags"] and len(e["exon_hashes"]) >= 3)

print(f"{example['transcript_id']}  ({example['gene_id']})")
print(f"protein: {len(example['protein_seq'])} aa   MD5={example['protein_md5']}   SQ={example['protein_sq']}")
print(f"CDS:     {len(example['cds_seq'])} nt   MD5={example['cds_md5']}   SQ={example['cds_sq']}")
for i, (m5, sq, n) in enumerate(example["exon_hashes"], 1):
    print(f"  exon {i:2d} ({n:4d} nt)  MD5={m5}  SQ={sq}")

ENST00000343518.11  (ENSG00000198062.16)
protein: 545 aa   MD5=dee9ad9a6daeee6433b36fbe4c02aea8   SQ=SQ.wXu0xTQ4jGmTyyF06KoXbdNhmWGEAjTU
CDS:     1635 nt   MD5=e43bdf9f06df41d32370cbbcf7b72332   SQ=SQ.SXbJWsDpP62uMsx_HrwcWv2evbszxSzQ
  exon  1 ( 632 nt)  MD5=968e6f4872cb5a5ec77622d27f4e1dec  SQ=SQ.Cybr_qEQu0q0etCI8MXMAyQW-ABEJoAv
  exon  2 ( 115 nt)  MD5=4e1040bca4bbff7a9a795631ddf576f0  SQ=SQ.mtfm__QLbRql-9QWG2W0aqybvZq0rkcr
  exon  3 ( 174 nt)  MD5=55beeea898745d220ba5a311e578dbaf  SQ=SQ.Ryr1LguasHBgRtwspSegkc7uiqH3Vnr7
  exon  4 ( 107 nt)  MD5=fe4374bf1031619332eb9b27b5d562a7  SQ=SQ.VVytbgwEgxI94QQi3AyQqzi5RTdfQHQL
  exon  5 ( 138 nt)  MD5=b42a0585c649f36a65fe1208c8ef0969  SQ=SQ.xItmqOD0wzKZqI1gGQlOr86iDQWreoWL
  exon  6 (  71 nt)  MD5=55d315a475d2cce9f34f5b7a6c25315d  SQ=SQ.kSvHZI-sZDPSV5_mD4cUxqIOEjgPG0__
  exon  7 (  71 nt)  MD5=c8af861297494208d9f9da63bdf39a62  SQ=SQ.hwJa7EfagvfVybwWZV6xypArrTWFfN9C
  exon  8 (  45 nt)  MD5=e31a4f7b182d959ecb93ccc3e679bb01  SQ=SQ.mlNFC5Yr8Uy6VdX

## Next steps

1. Persist `catalog` into the SQLite schema from the handoff (`hash_md5, hash_sq, seq_type, accession, gene_id,
   source, release, evidence, length, low_complexity_frac`) -- notebook `03_sqlite_catalog.ipynb`.
2. Add low-complexity flagging (`dustmasker`/`segmasker`) as metadata columns, not by altering hashed bytes.
3. Bring in the GIAB HG002/HG003/HG004 chr22 phased VCFs and repeat this extraction per-haplotype to get the
   trio inheritance comparison (Track 1).